# World Cup 2026 Winner Prediction

Project ini bertujuan untuk melakukan eksplorasi awal pada dataset pertandingan sepak bola internasional sebagai dasar pembuatan model prediksi juara World Cup 2026.


## Import Library


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

## Load Dataset

In [10]:
path = Path("../data/raw/results.csv")

df = pd.read_csv(path)

print("Dataset berhasil dibaca dari:", path)
df.head()

Dataset berhasil dibaca dari: ..\data\raw\results.csv


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


## Dataset Overview

In [13]:
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])

print("\nNama Kolom:")
print(df.columns.tolist())

print("\nInformasi dataset:")
df.info()

Jumlah baris: 49472
Jumlah kolom: 9

Nama Kolom:
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49472 entries, 0 to 49471
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49472 non-null  object 
 1   home_team   49472 non-null  object 
 2   away_team   49472 non-null  object 
 3   home_score  49400 non-null  float64
 4   away_score  49400 non-null  float64
 5   tournament  49472 non-null  object 
 6   city        49472 non-null  object 
 7   country     49472 non-null  object 
 8   neutral     49472 non-null  bool   
dtypes: bool(1), float64(2), object(6)
memory usage: 3.1+ MB


## Missing Value dan Data Duplikat

In [15]:
print("Jumlah missing value setiap kolom:")

miss_value = df.isnull().sum().sort_values(ascending=False)
display(miss_value)

print("\nJumlah data duplikat:", df.duplicated().sum())

Jumlah missing value setiap kolom:


away_score    72
home_score    72
date           0
away_team      0
home_team      0
tournament     0
city           0
country        0
neutral        0
dtype: int64


Jumlah data duplikat: 0


## Data Type Conversion

Pada bagian ini kolom tanggal diubah menjadi format datetime agar dapat digunakan untuk analisis berdasarkan tahun.

In [18]:
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

print("Tahun awal:", df["year"].min())
print("Tahun akhir:", df["year"].max())

df[["date", "year"]].head()

Tahun awal: 1872
Tahun akhir: 2026


,date,year
0,1872-11-30,1872
1,1873-03-08,1873
2,1874-03-07,1874
3,1875-03-06,1875
4,1876-03-04,1876


## Membuat Target Prediksi
Target prediksi dibuat berdasarkan skor pertandingan. Jika skor home team lebih besar, maka hasilnya home_win. Jika skor away team lebih besar, maka hasilnya away_win. Jika skor sama, maka hasilnya draw.

In [21]:
def get_result(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    elif row["home_score"] < row["away_score"]:
        return "away_win"
    else:
        return "draw"

df["result"] = df.apply(get_result, axis=1)
df[["home_team", "away_team", "home_score", "away_score", "result"]].head()

,home_team,away_team,home_score,away_score,result
0,Scotland,England,0.0,0.0,draw
1,England,Scotland,4.0,2.0,home_win
2,Scotland,England,2.0,1.0,home_win
3,England,Scotland,2.0,2.0,draw
4,Scotland,England,3.0,0.0,home_win


## Distribusi Hasil Pertandingan
Pada bagian ini dilakukan analisis jumlah hasil pertandingan berdasarkan kategori home_win, away_win, dan draw.

In [24]:
print("Distribusi hasil pertandingan:")

result_counts = df["result"].value_counts()
display(result_counts)

print("\nDistribusi hasil pertandingan dalam persen:")

result_percentege = (df["result"].value_counts(normalize=True) * 100).round(2)
display(result_percentege)

Distribusi hasil pertandingan:


result
home_win    24209
away_win    13958
draw        11305
Name: count, dtype: int64


Distribusi hasil pertandingan dalam persen:


result
home_win    48.93
away_win    28.21
draw        22.85
Name: proportion, dtype: float64

## Analisis Turnamen dan Tim
Pada bagian ini dilakukan pengecekan turnamen yang paling banyak muncul serta tim yang memiliki jumlah pertandingan terbanyak dalam dataset.

In [27]:
print("Turnamen dengan jumlah pertandingan terbanyak:")

tournament_counts = df["tournament"].value_counts().head(20)
display(tournament_counts)

print("\nTim dengan jumlah pertandingan terbanyak:")

all_teams = pd.concat([df["home_team"], df["away_team"]])
team_counts = all_teams.value_counts().head(20)

display(team_counts)

Turnamen dengan jumlah pertandingan terbanyak:


tournament
Friendly                                18384
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64


Tim dengan jumlah pertandingan terbanyak:


Sweden         1104
England        1092
Argentina      1072
Brazil         1062
Germany        1034
South Korea    1010
Hungary        1006
Mexico         1006
Uruguay         973
France          938
Italy           893
Poland          892
Switzerland     887
Netherlands     882
Norway          875
Denmark         874
Thailand        865
Austria         863
Belgium         856
Scotland        854
Name: count, dtype: int64

## Filter Data dan Menyimpan Data
Karena project ini bertujuan untuk memprediksi World Cup 2026, data pertandingan yang terlalu lama dapat kurang relevan. Oleh karena itu, dataset difilter mulai dari tahun 2000 keatas.

In [38]:
df_modern = df[df["year"] >= 2000].copy()

print("Jumlah data sebelum filter:", df.shape[0])
print("Jumlah data setelah filter:", df_modern.shape[0])
df_modern.head()

output_path= Path("../data/processed/match_processed.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_modern.to_csv(output_path, index=False)
print("Data berhasil disimpan ke:", output_path)

Jumlah data sebelum filter: 49472
Jumlah data setelah filter: 25410
Data berhasil disimpan ke: ..\data\processed\match_processed.csv
